# LPJ-GUESS → CLM `PCT_NAT_PFT`: corrected anomaly workflow

This notebook constructs a **semi-realistic, FPC-based natural-PFT redistribution** for CLM.

It deliberately does **not** assume that FPC is identical to PFT area. The preferred input is
peak-period monthly FPC, used as an area proxy after aggregating LPJ-GUESS PFTs within each
month. If actual crown-cover or PFT-area output becomes available, that should replace FPC.

Major safeguards relative to the earlier notebook:

- reconstruct the complete regular 0.5° source grid before conservative regridding;
- use an explicit source mask and preserve unmapped cells as missing;
- transfer the LPJ anomaly without forcing LPJ FPC into an artificial closed partition;
- apply losses first and scale **only positive increments** if available CLM area is insufficient;
- never silently replace missing anomalies with zero;
- quantify requested versus realised change and every constraint applied;
- write output only after validation and only when `WRITE_OUTPUT = True`.

**Interpretation:** because `PCT_NATVEG` is held fixed, the result is redistribution among natural
PFTs and natural bare ground—not a change in total natural-vegetation land-unit area.

## 1. Setup

Edit the paths and period labels below. Monthly files may contain either monthly climatologies
(`Lat`, `Lon`, `Month`) or monthly time series (`Lat`, `Lon`, `Year`, `Month`). Repeated header
rows are removed safely.

In [1]:
from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import cartopy.crs as ccrs

warnings.simplefilter("default")
xr.set_options(keep_attrs=True)
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.titlesize": 12,
})

In [4]:
# ------------------------------- paths ---------------------------------
DATA = Path("/nird/datapeak/NS9188K/adelez/BRL-FRST-XPSN/data/edit-surfdata")
SURF_IN = DATA / "surfdata_1.9x2.5_hist_78pfts_CMIP6_simyr2000_c190304.nc"

# Update these two names to the actual historical and future monthly files.
FPC_HIST = DATA / "LPJ-GUESS/mfpc_pft_avg_1971_2000_monthly_mean.out"
FPC_FUT = DATA / "LPJ-GUESS/mfpc_pft_avg_2070_2100_monthly_mean.out"

SURF_OUT = DATA / "surfdata_1.9x2.5_SSP585_2070-2100_peakFPC_LPJGUESS.nc"
FIGDIR = Path("figures_peak_fpc_surfdata")

HIST_PERIOD = "1971–2000"
FUT_PERIOD = "2071–2100"
COVER_METHOD = "peak"       # "peak" (recommended) or "mean" (sensitivity)
SOURCE_RESOLUTION = 0.5
WRITE_OUTPUT = False         # inspect diagnostics first, then set True
OVERWRITE = False
SAVE_FIGURES = True

for path in (SURF_IN, FPC_HIST, FPC_FUT):
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")
if COVER_METHOD not in {"peak", "mean"}:
    raise ValueError("COVER_METHOD must be 'peak' or 'mean'")
if SAVE_FIGURES:
    FIGDIR.mkdir(parents=True, exist_ok=True)

### PFT correspondence and assumptions

This preserves the mapping used in the original notebook so that the numerical correction is
separable from ecological mapping choices. The shrub, grass, nonvascular, and peatland choices
remain scientific assumptions and should be tested in sensitivity experiments.

In [5]:
LPJ_NATURAL = [
    "BNE", "BINE", "BNS", "TeNE", "TeBE", "IBS", "TeBS", "C3G",
    "HSE", "HSS", "LSE", "LSS", "GRT", "EPDS", "SPDS", "CLM",
]

# CLM natural-PFT index -> LPJ-GUESS PFT columns
CONVERSION_SCHEME = {
    1: ["TeNE"],                              # temperate needleleaf evergreen tree
    2: ["BNE", "BINE"],                       # boreal needleleaf evergreen tree
    3: ["BNS"],                               # boreal needleleaf deciduous tree
    5: ["TeBE"],                              # temperate broadleaf evergreen tree
    7: ["TeBS"],                              # temperate broadleaf deciduous tree
    8: ["IBS"],                               # boreal broadleaf deciduous tree
    11: ["HSE", "HSS", "LSE", "LSS", "EPDS", "SPDS"],
    12: ["C3G", "GRT"],
}

BARE = 0
PFTS_TO_CHANGE = sorted(CONVERSION_SCHEME)
CLM_NAMES = {
    0: "BG", 1: "TeNET", 2: "BoNET", 3: "BoNDT", 5: "TeBET",
    7: "TeBDT", 8: "BoBDT", 11: "BoBDS", 12: "aC3", 13: "C3",
}

mapping_rows = [
    {"CLM index": pft, "CLM name": CLM_NAMES[pft], "LPJ-GUESS inputs": ", ".join(cols)}
    for pft, cols in CONVERSION_SCHEME.items()
]
display(pd.DataFrame(mapping_rows))

,CLM index,CLM name,LPJ-GUESS inputs
0,1,TeNET,TeNE
1,2,BoNET,"BNE, BINE"
2,3,BoNDT,BNS
3,5,TeBET,TeBE
4,7,TeBDT,TeBS
5,8,BoBDT,IBS
6,11,BoBDS,"HSE, HSS, LSE, LSS, EPDS, SPDS"
7,12,aC3,"C3G, GRT"


## 2. Load and validate LPJ-GUESS data

Peak cover is calculated **after PFT aggregation within each month**. If `Year` is present, the
workflow calculates a peak for each year and then averages annual peaks. If only a 12-month
climatology is present, it takes the maximum monthly climatological value.

Peak values for different PFTs may occur in different months, so they are not summed and
interpreted as one simultaneous physical partition.

In [6]:
def _find_column(columns, candidates, required=True):
    lookup = {str(c).strip().casefold(): c for c in columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    if required:
        raise KeyError(f"Could not find any of {candidates}; columns are {list(columns)}")
    return None


def load_lpj_table(path):
    # Load whitespace output, remove repeated headers, and validate PFT values.
    raw = pd.read_csv(path, sep=r"\s+", comment="#", dtype=str)
    raw.columns = [str(c).strip() for c in raw.columns]
    lat_col = _find_column(raw.columns, ["Lat", "Latitude"])
    lon_col = _find_column(raw.columns, ["Lon", "Longitude"])
    year_col = _find_column(raw.columns, ["Year", "Yr"], required=False)
    month_col = _find_column(raw.columns, ["Month", "Mon"], required=False)

    rename = {lat_col: "lat", lon_col: "lon"}
    if year_col is not None:
        rename[year_col] = "year"
    if month_col is not None:
        rename[month_col] = "month"
    data = raw.rename(columns=rename)

    # Repeated header rows become NaN coordinates and are removed here.
    data["lat"] = pd.to_numeric(data["lat"], errors="coerce")
    data["lon"] = pd.to_numeric(data["lon"], errors="coerce")
    data = data.dropna(subset=["lat", "lon"]).copy()
    data["lon"] = ((data["lon"] + 180) % 360) - 180

    numeric = LPJ_NATURAL + [c for c in ("year", "month") if c in data]
    missing = sorted(set(LPJ_NATURAL) - set(data.columns))
    if missing:
        raise KeyError(f"Missing LPJ PFT columns in {path.name}: {missing}")
    for column in numeric:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    if data[LPJ_NATURAL].isna().any().any():
        bad = data.loc[data[LPJ_NATURAL].isna().any(axis=1)].head()
        raise ValueError(f"Non-numeric or missing PFT values in {path.name}:\n{bad}")
    if ((data[LPJ_NATURAL] < -1e-10) | (data[LPJ_NATURAL] > 1 + 1e-6)).any().any():
        raise ValueError(f"PFT FPC outside [0, 1] in {path.name}")
    if "month" in data and not data["month"].dropna().between(1, 12).all():
        raise ValueError(f"Invalid month values in {path.name}")

    keys = ["lat", "lon"] + [c for c in ("year", "month") if c in data]
    duplicates = data.duplicated(keys, keep=False)
    if duplicates.any():
        raise ValueError(f"Duplicate coordinate/time rows in {path.name}: {duplicates.sum()}")
    return data.sort_values(keys).reset_index(drop=True)


def aggregate_lpj_groups(data):
    keys = ["lat", "lon"] + [c for c in ("year", "month") if c in data]
    grouped = data[keys].copy()
    for clm_pft, lpj_columns in CONVERSION_SCHEME.items():
        grouped[f"pft_{clm_pft}"] = data[lpj_columns].sum(axis=1)
    return grouped


def period_cover(data, method):
    # Return one row per grid cell and mapped CLM group.
    grouped = aggregate_lpj_groups(data)
    pft_columns = [f"pft_{pft}" for pft in PFTS_TO_CHANGE]
    if method == "mean":
        return grouped.groupby(["lat", "lon"], as_index=False)[pft_columns].mean()
    if "month" not in grouped:
        raise ValueError("COVER_METHOD='peak' requires a Month/Mon column")
    if "year" in grouped:
        annual_peak = grouped.groupby(["lat", "lon", "year"], as_index=False)[pft_columns].max()
        return annual_peak.groupby(["lat", "lon"], as_index=False)[pft_columns].mean()
    return grouped.groupby(["lat", "lon"], as_index=False)[pft_columns].max()


hist_raw = load_lpj_table(FPC_HIST)
fut_raw = load_lpj_table(FPC_FUT)
hist_cover = period_cover(hist_raw, COVER_METHOD)
fut_cover = period_cover(fut_raw, COVER_METHOD)

hist_keys = pd.MultiIndex.from_frame(hist_cover[["lat", "lon"]])
fut_keys = pd.MultiIndex.from_frame(fut_cover[["lat", "lon"]])
if not hist_keys.equals(fut_keys):
    only_hist = len(hist_keys.difference(fut_keys))
    only_fut = len(fut_keys.difference(hist_keys))
    raise ValueError(f"Historical/future grids differ: hist-only={only_hist}, future-only={only_fut}")

input_summary = pd.DataFrame([
    {"period": HIST_PERIOD, "rows": len(hist_raw), "cells": len(hist_cover),
     "years": hist_raw.get("year", pd.Series(dtype=float)).nunique(),
     "months": hist_raw.get("month", pd.Series(dtype=float)).nunique()},
    {"period": FUT_PERIOD, "rows": len(fut_raw), "cells": len(fut_cover),
     "years": fut_raw.get("year", pd.Series(dtype=float)).nunique(),
     "months": fut_raw.get("month", pd.Series(dtype=float)).nunique()},
])
display(input_summary)

ValueError: Historical/future grids differ: hist-only=95, future-only=0

## 3. Reconstruct the complete regular LPJ grid

The point list is reindexed to a complete global 0.5° lattice. Missing gridlist cells remain
`NaN` and are represented by an explicit ESMF source mask. This prevents missing longitude or
latitude rows from being mistaken for oversized cells.

In [7]:
def regular_centres(resolution):
    lon = np.arange(-180 + resolution / 2, 180, resolution)
    lat = np.arange(-90 + resolution / 2, 90, resolution)
    return lat, lon


def assert_on_regular_grid(values, centres, resolution, name):
    index = (np.asarray(values) - centres[0]) / resolution
    mismatch = np.abs(index - np.rint(index))
    if mismatch.max(initial=0) > 1e-6:
        raise ValueError(f"{name} coordinates do not lie on the configured {resolution}° grid")


def cover_table_to_array(table, resolution=SOURCE_RESOLUTION):
    lat, lon = regular_centres(resolution)
    assert_on_regular_grid(table["lat"].unique(), lat, resolution, "latitude")
    assert_on_regular_grid(table["lon"].unique(), lon, resolution, "longitude")
    dataset = table.set_index(["lat", "lon"]).to_xarray().reindex(lat=lat, lon=lon)
    arrays = [dataset[f"pft_{pft}"].expand_dims(natpft=[pft]) for pft in PFTS_TO_CHANGE]
    return xr.concat(arrays, dim="natpft").transpose("natpft", "lat", "lon")


cover_hist_src = cover_table_to_array(hist_cover)
cover_fut_src = cover_table_to_array(fut_cover)
xr.align(cover_hist_src, cover_fut_src, join="exact")

source_mask = np.isfinite(cover_hist_src).all("natpft") & np.isfinite(cover_fut_src).all("natpft")
if not source_mask.any():
    raise ValueError("No common valid source cells")
delta_src = (cover_fut_src - cover_hist_src).where(source_mask)

source_qc = pd.DataFrame({
    "metric": [
        "valid source cells", "historical mapped-FPC sum > 1",
        "future mapped-FPC sum > 1", "largest historical mapped-FPC sum",
        "largest future mapped-FPC sum",
    ],
    "value": [
        int(source_mask.sum()),
        int(((cover_hist_src.sum("natpft") > 1) & source_mask).sum()),
        int(((cover_fut_src.sum("natpft") > 1) & source_mask).sum()),
        float(cover_hist_src.sum("natpft").where(source_mask).max()),
        float(cover_fut_src.sum("natpft").where(source_mask).max()),
    ],
})
display(source_qc)

,metric,value
0,valid source cells,16619.000000
1,historical mapped-FPC sum > 1,0.000000
2,future mapped-FPC sum > 1,145.000000
3,largest historical mapped-FPC sum,0.952115
4,largest future mapped-FPC sum,1.133245


## 4. Load CLM and conservatively regrid the anomaly

The anomaly is regridded directly. `unmapped_to_nan=True` makes source coverage explicit;
unmapped cells are not silently interpreted as zero change.

In [8]:
def convert_lsmcoord(ds):
    out = ds.copy()
    out = out.assign_coords(
        lsmlat=out["LATIXY"].isel(lsmlon=0).values,
        lsmlon=out["LONGXY"].isel(lsmlat=0).values,
    )
    return out.rename({"lsmlat": "lat", "lsmlon": "lon"})


def convert360_180(data):
    if float(data.lon.min()) >= 0:
        data = data.assign_coords(lon=((data.lon + 180) % 360) - 180).sortby("lon")
    return data


def convert180_360(data):
    if float(data.lon.min()) < 0:
        data = data.assign_coords(lon=data.lon % 360).sortby("lon")
    return data


def bounds_1d(values):
    values = np.asarray(values, dtype=float)
    spacing = np.diff(values)
    if not np.allclose(spacing, spacing[0], rtol=0, atol=1e-8):
        raise ValueError("Conservative regridding requires a complete regular coordinate axis")
    mid = 0.5 * (values[:-1] + values[1:])
    return np.r_[values[0] - spacing[0] / 2, mid, values[-1] + spacing[-1] / 2]


def make_grid(lat, lon, mask=None):
    grid = xr.Dataset({
        "lat": ("lat", np.asarray(lat)),
        "lon": ("lon", np.asarray(lon)),
        "lat_b": ("lat_b", bounds_1d(lat)),
        "lon_b": ("lon_b", bounds_1d(lon)),
    })
    if mask is not None:
        grid["mask"] = mask.astype(np.int32)
    return grid


surf = xr.open_dataset(SURF_IN)
surf_geo = convert_lsmcoord(surf)
pct_clm = convert360_180(surf_geo["PCT_NAT_PFT"])
landfrac = convert360_180(surf_geo["LANDFRAC_PFT"])
natveg = convert360_180(surf_geo["PCT_NATVEG"])

valid_clm = (
    (landfrac > 0) & (natveg > 0) &
    (np.abs(pct_clm.sum("natpft", skipna=False) - 100) < 1e-3)
).fillna(False)

src_grid = make_grid(delta_src.lat.values, delta_src.lon.values, source_mask)
dst_grid = make_grid(pct_clm.lat.values, pct_clm.lon.values)
regridder = xe.Regridder(
    src_grid, dst_grid, method="conservative_normed",
    unmapped_to_nan=True, ignore_degenerate=False,
)
delta_clm = regridder(delta_src, keep_attrs=True, skipna=True, na_thres=1.0)
target_coverage = np.isfinite(delta_clm).all("natpft")
modify_domain = valid_clm & target_coverage

if not modify_domain.any():
    raise ValueError("No CLM cells have both valid surfdata and LPJ anomaly coverage")
if delta_clm.where(modify_domain).isnull().any():
    raise ValueError("Missing anomaly inside the modification domain")

print(f"valid CLM cells: {int(valid_clm.sum())}")
print(f"cells with LPJ coverage: {int(target_coverage.sum())}")
print(f"cells to modify: {int(modify_domain.sum())}")

<frozen importlib._bootstrap>:488: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject
/cluster/home/adelez/nird/.conda/envs/lcc-env/lib/python3.12/site-packages/xesmf/backend.py:57: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')


ValueError: Missing anomaly inside the modification domain

## 5. Apply the anomaly with explicit constraints

Negative changes are applied first and clipped only where CLM contains too little of that PFT.
The resulting free area plus existing natural bare ground defines the capacity for gains. Only
positive increments are proportionally reduced when they exceed that capacity. Untouched PFTs
remain bit-identical.

In [ ]:
delta_requested = (100 * delta_clm).sel(natpft=PFTS_TO_CHANGE)  # percentage points
base_changed = pct_clm.sel(natpft=PFTS_TO_CHANGE)
untouched = [
    int(pft) for pft in pct_clm.natpft.values
    if int(pft) not in PFTS_TO_CHANGE and int(pft) != BARE
]
untouched_sum = pct_clm.sel(natpft=untouched).sum("natpft")
block_total = 100 - untouched_sum

requested_loss = xr.where(delta_requested < 0, delta_requested, 0)
requested_gain = xr.where(delta_requested > 0, delta_requested, 0)
after_loss = (base_changed + requested_loss).clip(min=0)
realised_loss = after_loss - base_changed

capacity_for_gain = (block_total - after_loss.sum("natpft")).clip(min=0)
gain_sum = requested_gain.sum("natpft")
gain_scale = xr.where(gain_sum > 0, np.minimum(1.0, capacity_for_gain / gain_sum), 1.0)
new_changed = after_loss + requested_gain * gain_scale
new_bare = block_total - new_changed.sum("natpft")

pct_new = pct_clm.copy(deep=True)
pct_new.loc[dict(natpft=PFTS_TO_CHANGE)] = new_changed
pct_new.loc[dict(natpft=BARE)] = new_bare
pct_new = xr.where(modify_domain, pct_new, pct_clm)

realised_delta = (pct_new - pct_clm).sel(natpft=PFTS_TO_CHANGE)
adjustment = realised_delta - delta_requested
loss_clipped = (requested_loss < realised_loss - 1e-10).any("natpft") & modify_domain
gain_scaled = (gain_scale < 1 - 1e-10) & modify_domain

print(f"cells with a clipped loss: {int(loss_clipped.sum())}")
print(f"cells with scaled positive gains: {int(gain_scaled.sum())}")
print(f"minimum gain scale: {float(gain_scale.where(modify_domain).min()):.4f}")

## 6. Validation and requested-versus-realised diagnostics

Physical-area summaries use gridcell area × land fraction × natural-vegetation fraction. This
is different from giving every natural-vegetation land unit equal weight.

In [ ]:
def spherical_cell_area(lat, lon, radius=6_371_000.0):
    lat_b = np.deg2rad(bounds_1d(lat))
    lon_b = np.deg2rad(bounds_1d(lon))
    values = radius**2 * np.diff(np.sin(lat_b))[:, None] * np.diff(lon_b)[None, :]
    return xr.DataArray(values, coords={"lat": lat, "lon": lon}, dims=("lat", "lon"))


def weighted_mean(field, weights, mask):
    usable = mask & np.isfinite(field) & np.isfinite(weights) & (weights > 0)
    return float((field.where(usable) * weights.where(usable)).sum() / weights.where(usable).sum())


# Core invariants
column_sum = pct_new.sum("natpft", skipna=False)
residual = np.abs(column_sum - 100).where(valid_clm)
assert float(residual.max(skipna=True)) < 1e-8
assert float(pct_new.where(valid_clm).min(skipna=True)) >= -1e-8
assert float(pct_new.where(valid_clm).max(skipna=True)) <= 100 + 1e-8
xr.testing.assert_identical(
    pct_new.sel(natpft=untouched),
    pct_clm.sel(natpft=untouched),
)
xr.testing.assert_identical(
    pct_new.where(~modify_domain),
    pct_clm.where(~modify_domain),
)

cell_area = spherical_cell_area(pct_clm.lat.values, pct_clm.lon.values)
physical_weight = cell_area * landfrac * natveg / 100

rows = []
for pft in PFTS_TO_CHANGE:
    requested = delta_requested.sel(natpft=pft)
    realised = realised_delta.sel(natpft=pft)
    rows.append({
        "PFT": CLM_NAMES[pft],
        "requested mean [pp]": weighted_mean(requested, physical_weight, modify_domain),
        "realised mean [pp]": weighted_mean(realised, physical_weight, modify_domain),
        "mean adjustment [pp]": weighted_mean(realised - requested, physical_weight, modify_domain),
        "max |adjustment| [pp]": float(np.abs(realised - requested).where(modify_domain).max()),
    })
change_summary = pd.DataFrame(rows)
display(change_summary.round(4))

assert np.isfinite(change_summary.select_dtypes("number").to_numpy()).all()
print(f"worst |PFT sum - 100|: {float(residual.max()):.3e}")

## 7. Maps

The third row is essential: it shows where feasibility constraints altered the requested LPJ
perturbation. A large or spatially coherent adjustment means the PFT mapping or allocation rule
needs revision before using the surfdata.

In [ ]:
def circle_boundary(ax):
    angle = np.linspace(0, 2 * np.pi, 100)
    vertices = np.column_stack([np.sin(angle), np.cos(angle)]) * 0.5 + 0.5
    ax.set_boundary(mpath.Path(vertices), transform=ax.transAxes)


def plot_map(field, ax, title, vmin=-40, vmax=40, cmap="BrBG"):
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    image = field.plot(
        ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(),
        add_colorbar=False, cmap=cmap, norm=norm,
    )
    ax.set_extent([-180, 180, 45, 90], ccrs.PlateCarree())
    circle_boundary(ax)
    ax.coastlines(linewidth=0.5)
    ax.gridlines(alpha=0.25)
    ax.set_title(title)
    return image


fields = []
titles = []
for label, data in [
    ("Requested", delta_requested),
    ("Realised", realised_delta),
    ("Adjustment", adjustment),
]:
    for pft in PFTS_TO_CHANGE:
        fields.append(data.sel(natpft=pft).where(modify_domain))
        titles.append(f"{label}: {CLM_NAMES[pft]}")

ncols = 4
nrows = int(np.ceil(len(fields) / ncols))
fig, axes = plt.subplots(
    nrows, ncols, figsize=(3.4 * ncols, 3.5 * nrows),
    subplot_kw={"projection": ccrs.Orthographic(0, 90)},
)
axes = np.asarray(axes).ravel()
image = None
for ax, field, title in zip(axes, fields, titles):
    image = plot_map(field, ax, title)
for ax in axes[len(fields):]:
    ax.axis("off")
fig.subplots_adjust(bottom=0.06, wspace=0.04, hspace=0.12)
colorbar = fig.colorbar(image, ax=axes.tolist(), orientation="horizontal", fraction=0.018, pad=0.02, extend="both")
colorbar.set_label("change in PCT_NAT_PFT [percentage points]")
if SAVE_FIGURES:
    fig.savefig(FIGDIR / "requested_realised_adjustment.png", dpi=220, bbox_inches="tight")
    fig.savefig(FIGDIR / "requested_realised_adjustment.pdf", bbox_inches="tight")
plt.show()

## 8. Colonization-threshold sensitivity

This diagnostic is descriptive, not proof that all threshold-crossing FPC change is area
expansion. It uses source-grid cell areas and reports several thresholds rather than selecting
one arbitrary value.

In [ ]:
source_area = spherical_cell_area(cover_hist_src.lat.values, cover_hist_src.lon.values)


def colonization_share(hist, fut, mask, threshold):
    change = (fut - hist).where(mask)
    positive = change > 0
    colonized = (hist < threshold) & (fut >= threshold)
    total_gain = (change.where(positive) * source_area).sum()
    colonized_gain = (change.where(positive & colonized) * source_area).sum()
    return float(100 * colonized_gain / total_gain) if float(total_gain) > 0 else np.nan


thresholds = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
rows = []
for threshold in thresholds:
    for pft in [2, 3, 8, 11, 12]:
        rows.append({
            "threshold": threshold,
            "PFT": CLM_NAMES[pft],
            "colonization share [%]": colonization_share(
                cover_hist_src.sel(natpft=pft), cover_fut_src.sel(natpft=pft),
                source_mask, threshold,
            ),
        })
colonization_sensitivity = pd.DataFrame(rows)
display(colonization_sensitivity.pivot(index="threshold", columns="PFT", values="colonization share [%]").round(1))

## 9. Write the validated surfdata

Set `WRITE_OUTPUT = True` only after inspecting:

1. source-grid QC;
2. number and location of constrained cells;
3. requested-versus-realised summary;
4. adjustment maps;
5. sensitivity to peak versus mean FPC and alternative PFT mappings.

The output is first written to a temporary file, reopened and validated, and then moved to the
requested path. Existing output is protected unless `OVERWRITE = True`.

In [ ]:
def sha256(path, chunk_size=1024**2):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_encoding(dataset):
    allowed = {
        "zlib", "complevel", "shuffle", "fletcher32", "contiguous", "chunksizes",
        "dtype", "_FillValue", "endian", "least_significant_digit", "significant_digits",
    }
    return {
        name: {key: value for key, value in variable.encoding.items() if key in allowed}
        for name, variable in dataset.variables.items()
    }


if WRITE_OUTPUT:
    if SURF_OUT.exists() and not OVERWRITE:
        raise FileExistsError(f"Output exists; set OVERWRITE=True to replace it: {SURF_OUT}")

    pct_native_geo = convert180_360(pct_new)
    original_geo = surf_geo["PCT_NAT_PFT"]
    if not np.allclose(pct_native_geo.lat, original_geo.lat):
        raise ValueError("Latitude order does not match the original surfdata")
    if not np.allclose(pct_native_geo.lon, original_geo.lon):
        raise ValueError("Longitude order does not match the original surfdata")
    if not np.array_equal(pct_native_geo.natpft, original_geo.natpft):
        raise ValueError("Natural-PFT coordinate does not match the original surfdata")

    surf_out = surf.copy(deep=True)
    replacement = xr.DataArray(
        pct_native_geo.transpose("natpft", "lat", "lon").values,
        dims=surf["PCT_NAT_PFT"].dims,
        coords=surf["PCT_NAT_PFT"].coords,
        attrs=surf["PCT_NAT_PFT"].attrs,
    )
    surf_out["PCT_NAT_PFT"] = replacement
    xr.testing.assert_identical(
        surf_out.drop_vars("PCT_NAT_PFT"),
        surf.drop_vars("PCT_NAT_PFT"),
    )

    provenance = {
        "method": f"{COVER_METHOD} FPC anomaly; losses first; positive gains scaled only if infeasible",
        "historical_period": HIST_PERIOD,
        "future_period": FUT_PERIOD,
        "historical_file": FPC_HIST.name,
        "future_file": FPC_FUT.name,
        "historical_sha256": sha256(FPC_HIST),
        "future_sha256": sha256(FPC_FUT),
        "source_resolution_degrees": SOURCE_RESOLUTION,
        "pft_mapping": CONVERSION_SCHEME,
        "cells_modified": int(modify_domain.sum()),
        "cells_loss_clipped": int(loss_clipped.sum()),
        "cells_gain_scaled": int(gain_scaled.sum()),
    }
    surf_out.attrs.update({
        "title": "CLM surfdata with LPJ-GUESS FPC-based natural-PFT redistribution",
        "source_vegetation": "LPJ-GUESS FPC",
        "history": (
            f"{dt.date.today():%Y-%m-%d}: PCT_NAT_PFT modified using {COVER_METHOD} "
            f"FPC anomaly ({FUT_PERIOD} minus {HIST_PERIOD}).\n" + surf.attrs.get("history", "")
        ),
        "comment": (
            "Only PCT_NAT_PFT changed. PCT_NATVEG and other land-unit fractions are fixed; "
            "therefore this is natural-PFT redistribution, not total natural-area change."
        ),
        "lpjguess_to_clm_provenance": json.dumps(provenance, sort_keys=True),
    })

    temporary = SURF_OUT.with_suffix(SURF_OUT.suffix + ".tmp")
    surf_out.to_netcdf(temporary, format="NETCDF4", encoding=safe_encoding(surf_out))
    with xr.open_dataset(temporary) as check:
        written_sum = check["PCT_NAT_PFT"].sum("natpft", skipna=False)
        assert float(np.abs(written_sum - 100).max()) < 1e-6
        assert not check["PCT_NAT_PFT"].isnull().any()
    os.replace(temporary, SURF_OUT)
    print(f"Saved and reopened successfully: {SURF_OUT}")
else:
    print("WRITE_OUTPUT=False: diagnostics completed; no surfdata file was written.")

## 10. Interpretation checklist

Before using the surface file in NorESM:

- run both `COVER_METHOD="peak"` and `"mean"` and compare the larch response;
- test at least one alternative shrub/grass mapping;
- investigate any coherent adjustment regions or many gain-scaled cells;
- document that FPC is an area proxy, not direct PFT area;
- spin up CLM-BGC with the modified surface distribution and inspect PFT carbon, LAI, height,
  albedo, evapotranspiration, and MEGAN emissions before coupling to CAM.